<a href="https://colab.research.google.com/github/asigatchov/vball-net-pytorch/blob/main/train_vball_net_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Setup


Clear the Colab workspace so the training environment starts from a clean state.


In [3]:
!rm -rf /content/*

Move into the Colab working directory, install `git`, and clone the repository.


In [4]:
%cd /content
!apt install git && git clone https://github.com/asigatchov/vball-net-pytorch.git
%cd /content/vball-net-pytorch

/content
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
Cloning into 'vball-net-pytorch'...
remote: Enumerating objects: 282, done.
remote: Counting objects: 100% (282/282), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 282 (delta 178), reused 197 (delta 100), pack-reused 0 (from 0)
Receiving objects: 100% (282/282), 6.69 MiB | 20.94 MiB/s, done.
Resolving deltas: 100% (178/178), done.
/content/vball-net-pytorch


## Extract the Dataset


Download the prepared volleyball dataset archive, extract it, and remove the compressed file.


In [5]:
!wget https://demo.vb-ai.ru/outputs/volleyball-split.tar
!tar -xf volleyball-split.tar
!rm volleyball-split.tar

--2026-06-10 11:07:45--  https://demo.vb-ai.ru/outputs/volleyball-split.tar
Resolving demo.vb-ai.ru (demo.vb-ai.ru)... 171.22.180.112
Connecting to demo.vb-ai.ru (demo.vb-ai.ru)|171.22.180.112|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 409927680 (391M) [application/x-tar]
Saving to: ‘volleyball-split.tar’

volleyball-split.ta 100%[===================>] 390.94M  11.4MB/s    in 34s     

2026-06-10 11:08:21 (11.4 MB/s) - ‘volleyball-split.tar’ saved [409927680/409927680]



Install PyTorch and the supporting Python packages required for data preparation and training.


In [6]:
!pip install -U pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install opencv-python pandas scipy tqdm tensorboard matplotlib seaborn requests av fvcore lion-pytorch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 71.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://download.pytorch.org/whl/cu124
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 59.4 MB/s  0:00:00
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61443 sha256=bf561581e036db3d24c2e65e058c423c1ccd370e0d25fb3122a07b59885de38a
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31596 sha256=bac7cb397f2a81e97d0

Define dataset paths and training parameters, including the checkpoint to resume from if needed.


In [16]:

from pathlib import Path
# Where to store the train/val/test split
SPLIT_DATA = "/content/vball-net-pytorch/volleyball-split"

# Where to store the prepared grid data
PREP_DATA = "/content/vball-net-pytorch/datasets"
OUTPUTS_DIR = "/content/vball-net-pytorch/outputs"

# Training mode:
# None -> train from scratch
# path to best.pth -> resume fine-tuning
RESUME_CKPT = "/content/drive/MyDrive/vball_work/outputs/VballNetGridV1b_seq9_grayscale_20260510_214641/checkpoints/best.pth"
# RESUME_CKPT = None

MODEL_NAME = "VballNetGridV1b"
SEQ = 9
GRAYSCALE = True

BATCH_SIZE = 8
WORKERS = 4
LR = 0.001

Convert the validation videos into grid-based heatmap tensors used by the model.


In [8]:
!python src/video_to_heatmap.py --source "$SPLIT_DATA/val"  --output "$PREP_DATA/val" \
  --mode grid \
  --force;

🏸 Badminton Dataset Preprocessor
📦 OpenCV 4.13.0
📦 NumPy 2.0.2
📦 Pandas 2.2.2
📂 Source: /content/vball-net-pytorch/volleyball-split/val
📂 Output: /content/vball-net-pytorch/datasets/val
🧩 Mode: grid
⏭️  Frame step: 1
OK: Found 9 match directories, 9 videos, 9 annotation files
Completed g_woman_transhmash_noisy_20260513: 1 sequences, 77 frames
Completed 4m2w_transmash_20250504: 1 sequences, 453 frames
Completed g_woman_pobeda_20251020_g1: 1 sequences, 202 frames
Completed g_beach_mix_20260507: 1 sequences, 643 frames
Completed g_4m2g_transhmash_20260424: 1 sequences, 205 frames
Completed g_man_novosil_20260328: 1 sequences, 561 frames
Completed g_4m2g_transhmash_20260508: 1 sequences, 326 frames
Completed g_woman_transhmash_20260401: 1 sequences, 109 frames
Completed beach_bl_night_20250825: 1 sequences, 74 frames
Processing frames: 100% 2650/2650 [01:48<00:00, 24.51frame/s]

Grid preprocessing completed
   Source: /content/vball-net-pytorch/volleyball-split/val
   Output: /content/vbal

Convert the training videos into grid-based heatmap tensors used during optimization.


In [9]:
!python src/video_to_heatmap.py --source "$SPLIT_DATA/train"  --output "$PREP_DATA/train" \
  --mode grid \
  --force;



🏸 Badminton Dataset Preprocessor
📦 OpenCV 4.13.0
📦 NumPy 2.0.2
📦 Pandas 2.2.2
📂 Source: /content/vball-net-pytorch/volleyball-split/train
📂 Output: /content/vball-net-pytorch/datasets/train
🧩 Mode: grid
⏭️  Frame step: 1
OK: Found 10 match directories, 54 videos, 54 annotation files
Completed g_woman_transhmash_noisy_20260513: 6 sequences, 1773 frames
Completed 4m2w_transmash_20250504: 4 sequences, 1079 frames
Completed g_woman_pobeda_20251020_g1: 4 sequences, 1745 frames
Completed g_man_beach_20260518: 4 sequences, 861 frames
Completed g_beach_mix_20260507: 6 sequences, 2018 frames
Completed g_4m2g_transhmash_20260424: 7 sequences, 2337 frames
Completed g_man_novosil_20260328: 5 sequences, 1575 frames
Completed g_4m2g_transhmash_20260508: 6 sequences, 1310 frames
Completed g_woman_transhmash_20260401: 6 sequences, 1161 frames
Completed beach_bl_night_20250825: 6 sequences, 1239 frames
Processing frames: 100% 15098/15098 [10:26<00:00, 24.10frame/s]

Grid preprocessing completed
   Sour

Print the full training command so the final configuration can be reviewed before launching training.


In [1]:
!echo src/train_grid.py \
  --data "$PREP_DATA/train" \
  --val_data "$PREP_DATA/val" \
  --model_name "$MODEL_NAME" \
  --seq "$SEQ" \
  --grayscale \
  --no-amp \
  --epochs "$EPOCHS" \
  --batch "$BATCH_SIZE" \
  --optimizer AdamW \
  --lr "$LR" \
  --workers "$WORKERS" \
  --out "$OUTPUTS_DIR" \
  --resume "$RESUME_CKPT"

src/train_grid.py --data /train --val_data /val --model_name  --seq  --grayscale --no-amp --epochs  --batch  --optimizer AdamW --lr  --workers  --out  --resume 


Start model training with the configured dataset paths, architecture, and optimization settings.


In [ ]:
!python src/train_grid.py --data "$PREP_DATA/train" --val_data "$PREP_DATA/val" \
--seq "$SEQ" --model_name "$MODEL_NAME" --grayscale --no-amp --batch "$BATCH_SIZE" \
 --optimizer AdamW --lr "$LR" --workers "$WORKERS"  --out "$OUTPUTS_DIR" --epochs 10

2026-06-10 11:34:42.846703: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Train samples: 7348
Val samples: 1292
train:   0%|                                                                | 0/919 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/datalo